In [11]:
from xarray_utils import analyze_netcdf, zarr_to_netcdf, find_missing_days
import pandas as pd
import xarray as xr
import numpy as np
import sklearn as sk
import sklearn as sk

In [12]:
# Open the Zarr dataset
ds_imd = xr.open_zarr("../data/raw/IMD_rainfall_0p25.zarr")
ds_imd = ds_imd.where(ds_imd != -999)

analyze_netcdf("../data/raw/IMD_rainfall_0p25.nc")


Analysis for NetCDF File: IMD_rainfall_0p25.nc

--- Dimensions ---
time: 31046
lat: 129
lon: 135

--- Coordinates ---
- lat:
    dtype: float64
    shape: (129,)
    attributes: {'axis': 'Y', 'long_name': 'latitude', 'standard_name': 'latitude', 'units': 'degrees_north'}
- lon:
    dtype: float64
    shape: (135,)
    attributes: {'axis': 'X', 'long_name': 'longitude', 'standard_name': 'longitude', 'units': 'degrees_east'}
- time:
    dtype: datetime64[ns]
    shape: (31046,)
    attributes: {'long_name': 'time', 'standard_name': 'time'}

--- Data Variables ---
- rain:
    dtype: float64
    shape: (31046, 129, 135)
    dimensions: ('time', 'lat', 'lon')
    attributes: {'long_name': 'Rainfall', 'units': 'mm/day'}

--- Global Attributes ---
Conventions: CF-1.7
comment: 
crs: epsg:4326
history: 2026-06-19 06:45:25.930728 Python
references: 
source: https://imdpune.gov.in/
title: IMD gridded data



In [13]:
# Open the Zarr dataset
ds_ecm = xr.open_zarr("../data/processed/s2s_reforecast_sorted.zarr")

analyze_netcdf("../data/raw/s2s_reforecast.nc")

Analysis for NetCDF File: s2s_reforecast.nc

--- Dimensions ---
time: 3720
step: 43
lat: 33
lon: 35

--- Coordinates ---
- lat:
    dtype: float64
    shape: (33,)
    attributes: {'long_name': 'latitude', 'standard_name': 'latitude', 'stored_direction': 'decreasing', 'units': 'degrees_north'}
- lon:
    dtype: float64
    shape: (35,)
    attributes: {'long_name': 'longitude', 'standard_name': 'longitude', 'units': 'degrees_east'}
- step:
    dtype: int64
    shape: (43,)
- time:
    dtype: datetime64[ns]
    shape: (3720,)
    attributes: {'long_name': 'initial time of forecast', 'standard_name': 'forecast_reference_time'}

--- Data Variables ---
- 10m_u_component_of_wind:
    dtype: float32
    shape: (3720, 43, 33, 35)
    dimensions: ('time', 'step', 'lat', 'lon')
    attributes: {'GRIB_NV': np.int64(0), 'GRIB_Nx': np.int64(35), 'GRIB_Ny': np.int64(33), 'GRIB_cfName': 'eastward_wind', 'GRIB_cfVarName': 'u10', 'GRIB_dataType': 'cf', 'GRIB_gridDefinitionDescription': 'Latitude/longi

In [4]:

# convert lead days into timedeltas
step_td = pd.to_timedelta(ds_ecm.step.values, unit="D").to_numpy()

ds_ecmv = ds_ecm.assign_coords(
    valid_time=(("time", "step"),
                ds_ecm.time.values[:, None] + step_td[None, :])
)


In [14]:
"""
WINDOWED correlation diagnostics for S2S.

Both predictors AND target are averaged over the same forecast window before
correlating. That is the whole point: independent day-to-day noise cancels in
each average, and only variability coherent across the window survives on both
sides to correlate. Windowing the target alone would mostly re-measure the
daily relationship.

Windows follow the S2S field standard (ECMWF operational products; Horat &
Lerch 2024 MWR; the ECMWF S2S AI Challenge), not arbitrary choices:

    week2      days  8-14    reference point
    week3-4    days 15-28    the headline S2S target
    week5-6    days 29-42    the hard one
    d14-42     days 14-42    your stated target window
    week3      days 15-21    finer resolution, if you want it
    week4      days 22-28
    week5      days 29-35
    week6      days 36-42

For each window: aggregate predictors over the window's leads, aggregate IMD
rain over the SAME valid dates, de-climatologise both, correlate per cell.

Anomalies are mandatory here. Windowed RAW values correlate at r ~ 0.9 for
almost everything -- window-averaging strengthens the seasonal cycle (it is
coherent across the window by construction) while cancelling weather noise.
Raw windowed correlation is the artefact at its most flattering.

Climatology is fit on TRAIN inits only. Same leakage rule as the baselines.
"""

import os
import time
from contextlib import contextmanager

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from dask.diagnostics import ProgressBar

# ---- CONFIG ----
IMD_TARGET_VAR = "rain"
DTYPE = np.float32
COARSE_PAD = 2.0
MONTHS = None                   # e.g. (6, 7, 8, 9) for JJAS inits only
CLIM_WINDOW_DAYS = 7
OUTDIR = "../results/diagnostics/corr_windowed"

# (name, first_lead_day, last_lead_day) inclusive
WINDOWS = [
    ("week2",   8, 14),
    ("week3-4", 15, 28),
    ("week5-6", 29, 42),
    ("d14-42",  14, 42),
    ("week3",   15, 21),
    ("week4",   22, 28),
    ("week5",   29, 35),
    ("week6",   36, 42),
]

WINDOW_PREDICTORS = True        # False -> window target only (the weaker variant)
COMPARE_DAILY_LEAD = 22         # single lead to plot alongside, for contrast

ZONE_SHAPEFILE = None
LAT_CUT, LON_CUT = 23.5, 82.5   # Tropic of Cancer / India's standard meridian
ZONES = [
    ("NW", lambda la, lo: (la >= LAT_CUT) & (lo < LON_CUT)),
    ("NE", lambda la, lo: (la >= LAT_CUT) & (lo >= LON_CUT)),
    ("SW", lambda la, lo: (la < LAT_CUT) & (lo < LON_CUT)),
    ("SE", lambda la, lo: (la < LAT_CUT) & (lo >= LON_CUT)),
]


@contextmanager
def stage(name):
    print(f"[ ] {name} ...", flush=True)
    t0 = time.perf_counter()
    yield
    print(f"[x] {name}  ({time.perf_counter() - t0:.1f}s)", flush=True)


# ---------------- plumbing ----------------

def normalize_step(ds, verbose=True):
    v = ds["step"].values
    if np.issubdtype(v.dtype, np.timedelta64):
        return ds
    mx = int(np.nanmax(v))
    u = "D" if mx <= 60 else ("h" if mx <= 24 * 60 else "s")
    if verbose:
        print(f"    step is {v.dtype} (max {mx}) -> units='{u}'")
    td = v.astype(np.int64).astype(f"timedelta64[{u}]").astype("timedelta64[ns]")
    return ds.assign_coords(step=("step", td))


def ensure_valid_time(ds, verbose=True):
    ds = normalize_step(ds, verbose=verbose)
    expected = ds["time"] + ds["step"]
    if "valid_time" in ds.coords:
        got = ds["valid_time"]
        if got.shape == expected.shape and (got.values == expected.values).all():
            return ds
        if verbose:
            print("    WARNING: existing valid_time != time + step -> rebuilding")
        ds = ds.drop_vars("valid_time")
    return ds.assign_coords(valid_time=expected)


def build_cells(imd_ds, var=IMD_TARGET_VAR):
    mask2d = imd_ds[var].notnull().any(dim="time").compute()
    st = mask2d.stack(cell=("lat", "lon"))
    cells = st[st.values]
    print(f"    {len(cells['cell'])}/{mask2d.size} valid cells")
    return cells


def load_coarse(ecmwf_ds, imd_ds, months=MONTHS, pad=COARSE_PAD):
    ecmwf_ds = ensure_valid_time(ecmwf_ds)
    for c in ("lat", "lon"):
        if ecmwf_ds[c].values[0] > ecmwf_ds[c].values[-1]:
            ecmwf_ds = ecmwf_ds.sortby(c)
    lat0, lat1 = float(imd_ds.lat.min()), float(imd_ds.lat.max())
    lon0, lon1 = float(imd_ds.lon.min()), float(imd_ds.lon.max())
    sub = ecmwf_ds.sel(lat=slice(lat0 - pad, lat1 + pad),
                       lon=slice(lon0 - pad, lon1 + pad))
    if months is not None:
        sub = sub.sel(time=sub["time"].dt.month.isin(list(months)))
    print(f"    {dict(sub.sizes)} x {len(sub.data_vars)} vars")
    with ProgressBar():
        return sub.astype(DTYPE).compute()


def assign_zones(cells):
    lat, lon = cells["lat"].values, cells["lon"].values
    z = np.empty(len(lat), dtype=object)
    un = np.ones(len(lat), bool)
    for name, rule in ZONES:
        m = rule(lat, lon) & un
        z[m] = name
        un &= ~m
    return z


def _doy_matrix(doys, window, n_doy=366):
    centers = np.arange(1, n_doy + 1)
    d = np.abs(doys[None, :].astype(int) - centers[:, None])
    return (np.minimum(d, n_doy - d) <= window).astype(DTYPE)


def _clim_grid(values, doys, window):
    """(n, c) -> (366, c). NaN-aware without ever building (366, n, c)."""
    M = _doy_matrix(doys, window)
    counts = M @ np.isfinite(values).astype(DTYPE)
    sums = M @ np.nan_to_num(values).astype(DTYPE)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(counts > 0, sums / np.maximum(counts, 1), np.nan).astype(DTYPE)


def corr_cells(X, y):
    """X (n, c, f), y (n, c) -> (c, f) Pearson r per cell per variable."""
    Xc = X - np.nanmean(X, axis=0)
    yc = y - np.nanmean(y, axis=0)
    num = np.nansum(Xc * yc[:, :, None], axis=0)
    den = np.sqrt(np.nansum(Xc ** 2, axis=0) * np.nansum(yc ** 2, axis=0)[:, None])
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(den > 0, num / den, np.nan)


def corr_1d(x, y):
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 3:
        return np.nan
    a, b = x[m] - x[m].mean(), y[m] - y[m].mean()
    d = np.sqrt((a ** 2).sum() * (b ** 2).sum())
    return float((a * b).sum() / d) if d > 0 else np.nan


# ---------------- windowed aggregation ----------------

def window_slice(coarse, imd_cells, lo, hi, lat_pts, lon_pts, feature_vars,
                 window_predictors=WINDOW_PREDICTORS):
    """Aggregate predictors and target over lead days [lo, hi] inclusive.

    Predictors: mean over the window's leads, then interp to valid cells.
    Averaging on the COARSE grid first (before interp) is exact and ~6x
    cheaper -- both operations are linear, so they commute.

    Target: mean of IMD rain over the SAME valid dates the window spans, i.e.
    init+lo .. init+hi. This is the bit that must line up; getting the target
    window from anything other than the forecast's own valid dates silently
    correlates against the wrong fortnight.
    """
    leads = (coarse["step"].values / np.timedelta64(1, "D")).astype(int)
    sel = np.where((leads >= lo) & (leads <= hi))[0]
    if len(sel) == 0:
        raise ValueError(f"no leads in [{lo}, {hi}]; have {leads.min()}..{leads.max()}")

    if window_predictors:
        agg = coarse.isel(step=sel).mean(dim="step")
    else:
        mid = sel[len(sel) // 2]
        agg = coarse.isel(step=mid)

    X = np.stack([agg[v].interp(lat=lat_pts, lon=lon_pts).values
                  for v in feature_vars], axis=-1).astype(DTYPE)

    # target: average IMD over each init's own window of valid dates
    vt = coarse["valid_time"].isel(step=sel).dt.floor("D").values   # (n_init, n_lead)
    flat = vt.ravel()
    y_all = imd_cells.reindex(time=flat).values                     # (n_init*n_lead, c)
    y = np.nanmean(y_all.reshape(vt.shape + (-1,)), axis=1)         # (n_init, c)

    # label the window by its centre valid date, for climatology lookup
    centre = coarse["time"].values + np.timedelta64((lo + hi) // 2, "D")
    doy = xr.DataArray(centre, dims="t").dt.dayofyear.values
    return X, y, doy, len(sel)


# ---------------- plots ----------------

def boxplot_window(r, feature_vars, wname, lo, hi, n_leads, outdir):
    order = np.argsort(-np.nanmedian(np.abs(r), axis=0))
    data = [r[:, k][np.isfinite(r[:, k])] for k in order]
    labels = [feature_vars[k] for k in order]

    fig, ax = plt.subplots(figsize=(13, 6))
    bp = ax.boxplot(data, showfliers=False, patch_artist=True, whis=(5, 95))
    for p in bp["boxes"]:
        p.set_facecolor("#E45756")
        p.set_alpha(0.7)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_xticks(range(1, len(labels) + 1))
    ax.set_xticklabels(labels, rotation=90, fontsize=8)
    ax.set_ylabel("Pearson r vs windowed rain anomaly (per cell)")
    ax.set_ylim(-1, 1)
    ax.set_title(f"{wname}  (days {lo}-{hi}, {n_leads} leads averaged)  --  "
                 f"anomaly space, {len(data[0])} cells")
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(outdir, f"corr_box_{wname}.png"), dpi=110)
    plt.close(fig)


def lift_plot(med_win, med_daily, wnames, feature_vars, outdir, daily_lead):
    """The money plot: windowed vs single-day correlation, same predictors.
    If windowing does nothing, these overlap and the premise was wrong."""
    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(feature_vars))
    order = np.argsort(-med_win[wnames.index("week3-4")])
    for wi, w in enumerate(["week2", "week3-4", "week5-6"]):
        ax.plot(x, med_win[wnames.index(w)][order], "o-", lw=1.5, ms=4, label=w)
    ax.plot(x, med_daily[order], "k--", lw=1.2, ms=3,
            label=f"single day (lead {daily_lead})")
    ax.set_xticks(x)
    ax.set_xticklabels([feature_vars[k] for k in order], rotation=90, fontsize=8)
    ax.set_ylabel("median |r| across cells")
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    ax.legend()
    ax.set_title("Does window-averaging lift correlation? "
                 "(both sides windowed, anomaly space)")
    fig.tight_layout()
    fig.savefig(os.path.join(outdir, "windowed_vs_daily.png"), dpi=130)
    plt.close(fig)


def zone_heatmap(zr, zone_names, feature_vars, wnames, outdir):
    fig, axes = plt.subplots(1, len(zone_names),
                             figsize=(4.2 * len(zone_names), 7), sharey=True)
    axes = np.atleast_1d(axes)
    for zi, (ax, zn) in enumerate(zip(axes, zone_names)):
        im = ax.imshow(zr[:, zi, :].T, aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1)
        ax.set_xticks(range(len(wnames)))
        ax.set_xticklabels(wnames, rotation=90, fontsize=8)
        ax.set_yticks(range(len(feature_vars)))
        ax.set_yticklabels(feature_vars, fontsize=7)
        ax.set_title(zn)
    fig.colorbar(im, ax=axes, label="r (zone-mean anomaly, windowed)")
    fig.savefig(os.path.join(outdir, "zone_corr_windowed.png"), dpi=130,
                bbox_inches="tight")
    plt.close(fig)


# ---------------- driver ----------------

if __name__ == "__main__":
    ecmwf_ds = ds_ecmv   # noqa: F821
    imd_ds = ds_imd      # noqa: F821

    os.makedirs(OUTDIR, exist_ok=True)
    t0 = time.perf_counter()

    with stage("Setup"):
        cells = build_cells(imd_ds)
        lat_pts = xr.DataArray(cells["lat"].values, dims="cell")
        lon_pts = xr.DataArray(cells["lon"].values, dims="cell")
        zlab = assign_zones(cells)
        zone_names = [z for z, _ in ZONES]
        for z in zone_names:
            print(f"    zone {z}: {(zlab == z).sum()} cells")

    with stage("Loading coarse ECMWF"):
        coarse = load_coarse(ecmwf_ds, imd_ds)
        feature_vars = list(coarse.data_vars)

    with stage("Loading IMD"):
        imd_cells = imd_ds[IMD_TARGET_VAR].stack(cell=("lat", "lon")).sel(
            cell=cells["cell"]).astype(DTYPE)
        with ProgressBar():
            imd_cells = imd_cells.compute()
        imd_cells = imd_cells.assign_coords(time=imd_cells["time"].dt.floor("D"))

    yrs = sorted(set(coarse["time"].dt.year.values.tolist()))
    tr_i = ~coarse["time"].dt.year.isin(yrs[-6:]).values   # climatology: train only
    print(f"    climatology from {tr_i.sum()} train inits "
          f"(holding out {yrs[-6:]})")

    n_f, n_z = len(feature_vars), len(zone_names)
    wnames = [w[0] for w in WINDOWS]
    med_win = np.full((len(WINDOWS), n_f), np.nan)
    zr = np.full((len(WINDOWS), n_z, n_f), np.nan)
    rows = []

    with stage(f"Windowed correlations: {len(WINDOWS)} windows x {n_f} vars"):
        for wi, (wname, lo, hi) in enumerate(WINDOWS):
            X, y, doy, n_leads = window_slice(
                coarse, imd_cells, lo, hi, lat_pts, lon_pts, feature_vars)

            # de-climatologise both sides. window index plays the role lead
            # played before: the model's drift is a property of the window.
            co = _clim_grid(y[tr_i], doy[tr_i], CLIM_WINDOW_DAYS)
            y = y - co[doy - 1]
            for k in range(n_f):
                cm = _clim_grid(X[tr_i, :, k], doy[tr_i], CLIM_WINDOW_DAYS)
                X[:, :, k] -= cm[doy - 1]

            r = corr_cells(X, y)
            med_win[wi] = np.nanmedian(np.abs(r), axis=0)
            boxplot_window(r, feature_vars, wname, lo, hi, n_leads, OUTDIR)

            for zi, z in enumerate(zone_names):
                m = (zlab == z)
                yz = np.nanmean(y[:, m], axis=1)
                for k in range(n_f):
                    zr[wi, zi, k] = corr_1d(np.nanmean(X[:, m, k], axis=1), yz)

            best = int(np.nanargmax(med_win[wi]))
            rows.append((wname, lo, hi, n_leads, np.nanmedian(med_win[wi]),
                         med_win[wi][best], feature_vars[best]))
            print(f"    {wname:>8} (d{lo}-{hi}, {n_leads:>2} leads): "
                  f"median |r| {np.nanmedian(med_win[wi]):.3f}, "
                  f"best {med_win[wi][best]:.3f} ({feature_vars[best]})", flush=True)
            del X, y

    with stage(f"Single-day reference (lead {COMPARE_DAILY_LEAD})"):
        Xd, yd, dd, _ = window_slice(coarse, imd_cells, COMPARE_DAILY_LEAD,
                                     COMPARE_DAILY_LEAD, lat_pts, lon_pts,
                                     feature_vars)
        co = _clim_grid(yd[tr_i], dd[tr_i], CLIM_WINDOW_DAYS)
        yd = yd - co[dd - 1]
        for k in range(n_f):
            cm = _clim_grid(Xd[tr_i, :, k], dd[tr_i], CLIM_WINDOW_DAYS)
            Xd[:, :, k] -= cm[dd - 1]
        med_daily = np.nanmedian(np.abs(corr_cells(Xd, yd)), axis=0)

    with stage("Figures"):
        lift_plot(med_win, med_daily, wnames, feature_vars, OUTDIR,
                  COMPARE_DAILY_LEAD)
        zone_heatmap(zr, zone_names, feature_vars, wnames, OUTDIR)
        xr.Dataset(
            {"median_abs_r": (("window", "var"), med_win),
             "zone_r": (("window", "zone", "var"), zr),
             "median_abs_r_daily": (("var",), med_daily)},
            coords={"window": wnames, "var": feature_vars, "zone": zone_names},
        ).to_netcdf(os.path.join(OUTDIR, "correlations_windowed.nc"))

    print(f"\nTotal: {time.perf_counter() - t0:.1f}s\n")
    print(f"{'window':>8} {'days':>8} {'leads':>6} {'med|r|':>7} {'best|r|':>8}  best var")
    for w, lo, hi, nl, med, bst, bv in rows:
        print(f"{w:>8} {f'{lo}-{hi}':>8} {nl:>6} {med:>7.3f} {bst:>8.3f}  {bv}")
    print(f"\nsingle day (lead {COMPARE_DAILY_LEAD}): median |r| "
          f"{np.nanmedian(med_daily):.3f}, best {med_daily.max():.3f}")
    print("\n-> windowed_vs_daily.png is the plot that answers the question.")
    print("   If week3-4 sits on top of the single-day line, windowing bought")
    print("   nothing and the collapse is real, not a noise artefact.")

[ ] Setup ...
    4964/17415 valid cells
    zone NW: 1874 cells
    zone NE: 828 cells
    zone SW: 1852 cells
    zone SE: 410 cells
[x] Setup  (2.7s)
[ ] Loading coarse ECMWF ...
    step is int64 (max 42) -> units='D'
    {'time': 3720, 'step': 43, 'lat': 33, 'lon': 35} x 21 vars
[########################################] | 100% Completed | 17.69 s
[x] Loading coarse ECMWF  (18.0s)
[ ] Loading IMD ...
[########################################] | 100% Completed | 4.14 sms
[x] Loading IMD  (4.9s)
    climatology from 2604 train inits (holding out [2019, 2020, 2021, 2022, 2023, 2024])
[ ] Windowed correlations: 8 windows x 21 vars ...
       week2 (d8-14,  7 leads): median |r| 0.092, best 0.294 (total_precipitation)
     week3-4 (d15-28, 14 leads): median |r| 0.051, best 0.126 (total_precipitation)
     week5-6 (d29-42, 14 leads): median |r| 0.024, best 0.053 (total_precipitation)
      d14-42 (d14-42, 29 leads): median |r| 0.045, best 0.095 (total_precipitation)
       week3 (d15-21,

In [15]:
da = imd_ds["rain"]
print(int(da.notnull().any(dim="time").sum()), int(da.notnull().all(dim="time").sum()))

4964 4885


In [23]:
bad = da.notnull().any("time") & ~da.notnull().all("time")
bad = (da.notnull().any("time") & ~da.notnull().all("time")).compute()
badd= [(bad.where(bad, drop=True).coords)]
badd

[Coordinates:
   * lat      (lat) float64 120B 21.0 21.25 21.5 21.75 ... 23.75 24.0 24.25 24.5
   * lon      (lon) float64 80B 68.0 68.25 68.5 68.75 ... 69.5 69.75 70.0 70.25]

In [10]:
"""
Diagnostic 1: per-lead box plots of variable-vs-rain correlation, distribution
              taken across the 4964 valid IMD cells (one box per predictor).
Diagnostic 2: the same, stratified into four regional zones, plus the
              inter-zone rain correlation matrix.


READ THIS FIRST: CORRELATIONS ARE COMPUTED ON ANOMALIES
-------------------------------------------------------
Correlating RAW predictor against RAW rain gives r ~ 0.8 for almost every
variable at almost every lead, because both carry the monsoon annual cycle.
That is the same artefact that made a raw linear regression look like it had
corr 0.84 skill at day 42 when its true skill was ~0.01. Every box plot would
look magnificent and mean nothing.

So: each side is de-climatologised first. Predictors lose a LEAD-DEPENDENT
model climatology (drift grows with lead); rain loses a DOY-only observed
climatology. Climatology is fit on TRAIN inits only, so these plots stay
honest if you later reuse them next to skill numbers.

Set ANOMALY = False only if you specifically want to see the artefact.


ON THE FOUR ZONES -- THE HONEST VERSION
---------------------------------------
IMD's four homogeneous regions (NWI, CI, NEI, SPIN) are NOT lat/lon boxes.
Parthasarathy et al. (1995) define them as GROUPS OF METEOROLOGICAL
SUBDIVISIONS -- polygons following state/district boundaries. Zheng et al.
(2016, JGR, 10.1002/2016JD025135), doing this exact projection, states that
gridded rainfall "can be projected only approximately onto the four IMD
regions due primarily to the challenge of exactly delineating the complex
boundary between meteorological subdivisions by grid box."

Therefore ZONES_APPROX below is an APPROXIMATION, not a literature standard.
Do not report it as "the IMD homogeneous regions". Either:
  (a) call it "approximate zones following Zheng et al. (2016)" and cite that
      the projection is approximate, or
  (b) set ZONE_SHAPEFILE to IMD's 36-subdivision shapefile and group them --
      the correct, citable route. assign_zones() supports both.
"""

import os
import time
from contextlib import contextmanager

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from dask.diagnostics import ProgressBar

# ---- CONFIG ----
IMD_TARGET_VAR = "rain"
ANOMALY = True                  # False reproduces the seasonal-cycle artefact
CLIM_WINDOW_DAYS = 0
DTYPE = np.float32
COARSE_PAD = 2.0
LEADS = None                    # e.g. range(14, 43); None = all
MONTHS = None                   # e.g. (6, 7, 8, 9) for JJAS
OUTDIR = "../results/diagnostics/corr_diagnostics_3"

LAT_CUT, LON_CUT = 23.5, 82.5

ZONES_APPROX = [
    ("NW", lambda la, lo: (la >= LAT_CUT) & (lo <  LON_CUT)),
    ("NE", lambda la, lo: (la >= LAT_CUT) & (lo >= LON_CUT)),
    ("SW", lambda la, lo: (la <  LAT_CUT) & (lo <  LON_CUT)),
    ("SE", lambda la, lo: (la <  LAT_CUT) & (lo >= LON_CUT)),
]


@contextmanager
def stage(name):
    print(f"[ ] {name} ...", flush=True)
    t0 = time.perf_counter()
    yield
    print(f"[x] {name}  ({time.perf_counter() - t0:.1f}s)", flush=True)


# ---------------- shared plumbing (same as the baseline scripts) ----------------

def normalize_step(ds, verbose=True):
    v = ds["step"].values
    if np.issubdtype(v.dtype, np.timedelta64):
        return ds
    mx = int(np.nanmax(v))
    u = "D" if mx <= 60 else ("h" if mx <= 24 * 60 else "s")
    if verbose:
        print(f"    step is {v.dtype} (max {mx}) -> units='{u}'")
    td = v.astype(np.int64).astype(f"timedelta64[{u}]").astype("timedelta64[ns]")
    return ds.assign_coords(step=("step", td))


def ensure_valid_time(ds, verbose=True):
    ds = normalize_step(ds, verbose=verbose)
    expected = ds["time"] + ds["step"]
    if "valid_time" in ds.coords:
        got = ds["valid_time"]
        if got.shape == expected.shape and (got.values == expected.values).all():
            return ds
        if verbose:
            print("    WARNING: existing valid_time != time + step -> rebuilding")
        ds = ds.drop_vars("valid_time")
    return ds.assign_coords(valid_time=expected)


def build_cells(imd_ds, var=IMD_TARGET_VAR):
    mask2d = imd_ds[var].notnull().any(dim="time").compute()
    stacked = mask2d.stack(cell=("lat", "lon"))
    cells = stacked[stacked.values]
    print(f"    {len(cells['cell'])} valid cells")
    return cells


def load_coarse(ecmwf_ds, imd_ds, months=MONTHS, leads=LEADS, pad=COARSE_PAD):
    ecmwf_ds = ensure_valid_time(ecmwf_ds)
    for c in ("lat", "lon"):
        if ecmwf_ds[c].values[0] > ecmwf_ds[c].values[-1]:
            ecmwf_ds = ecmwf_ds.sortby(c)
    lat0, lat1 = float(imd_ds.lat.min()), float(imd_ds.lat.max())
    lon0, lon1 = float(imd_ds.lon.min()), float(imd_ds.lon.max())
    sub = ecmwf_ds.sel(lat=slice(lat0 - pad, lat1 + pad), lon=slice(lon0 - pad, lon1 + pad))
    if months is not None:
        sub = sub.sel(time=sub["time"].dt.month.isin(list(months)))
    if leads is not None:
        sub = sub.isel(step=list(leads))
    with ProgressBar():
        return sub.astype(DTYPE).compute()


def _doy_matrix(doys, window, n_doy=366):
    centers = np.arange(1, n_doy + 1)
    d = np.abs(doys[None, :].astype(int) - centers[:, None])
    return (np.minimum(d, n_doy - d) <= window).astype(DTYPE)


def _clim_grid(values, doys, window):
    """(n, c) -> (366, c), NaN-aware via matmul counts."""
    M = _doy_matrix(doys, window)
    counts = M @ np.isfinite(values).astype(DTYPE)
    sums = M @ np.nan_to_num(values).astype(DTYPE)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(counts > 0, sums / np.maximum(counts, 1), np.nan).astype(DTYPE)


# ---------------- zones ----------------

# def assign_zones(cells, shapefile=ZONE_SHAPEFILE):
#     """Return (n_cell,) array of zone labels.

#     shapefile route = correct (IMD's actual subdivision polygons).
#     box route = approximation; label it as such in any figure caption.
#     """
#     lat = cells["lat"].values
#     lon = cells["lon"].values

#     if shapefile is not None:
#         import geopandas as gpd
#         from shapely.geometry import Point
#         gdf = gpd.read_file(shapefile)
#         pts = gpd.GeoDataFrame(geometry=[Point(x, y) for x, y in zip(lon, lat)],
#                                crs=gdf.crs)
#         joined = gpd.sjoin(pts, gdf, how="left", predicate="within")
#         # NOTE: you must map subdivision names -> NWI/CI/NEI/SPIN per
#         # Parthasarathy et al. (1995) Table 1. Column name varies by file.
#         return joined.iloc[:, -1].values

#     z = np.empty(len(lat), dtype=object)
#     unassigned = np.ones(len(lat), bool)
#     for name, rule in ZONES_APPROX:
#         m = rule(lat, lon) & unassigned
#         z[m] = name
#         unassigned &= ~m
#     return z


# ---------------- correlation ----------------

def corr_cells(X, y):
    """X (n, c, f), y (n, c) -> (c, f) Pearson r per cell per variable."""
    Xc = X - np.nanmean(X, axis=0)
    yc = y - np.nanmean(y, axis=0)
    num = np.nansum(Xc * yc[:, :, None], axis=0)
    den = np.sqrt(np.nansum(Xc ** 2, axis=0) * np.nansum(yc ** 2, axis=0)[:, None])
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(den > 0, num / den, np.nan)


def corr_1d(x, y):
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 3:
        return np.nan
    a, b = x[m] - x[m].mean(), y[m] - y[m].mean()
    d = np.sqrt((a ** 2).sum() * (b ** 2).sum())
    return float((a * b).sum() / d) if d > 0 else np.nan


# ---------------- plotting ----------------

def boxplot_lead(r, feature_vars, lead, outdir, anomaly):
    """r: (c, f) per-cell correlations. One box per variable."""
    order = np.argsort(-np.nanmedian(np.abs(r), axis=0))
    data = [r[:, k][np.isfinite(r[:, k])] for k in order]
    labels = [feature_vars[k] for k in order]

    fig, ax = plt.subplots(figsize=(13, 6))
    bp = ax.boxplot(data, showfliers=False, patch_artist=True, whis=(5, 95))
    for p in bp["boxes"]:
        p.set_facecolor("#4C78A8")
        p.set_alpha(0.65)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_xticks(range(1, len(labels) + 1))
    ax.set_xticklabels(labels, rotation=90, fontsize=8)
    ax.set_ylabel("Pearson r vs rain (per cell)")
    ax.set_ylim(-1, 1)
    kind = "anomaly" if anomaly else "RAW (seasonal cycle included!)"
    ax.set_title(f"Lead {lead} d  --  {kind} space, "
                 f"distribution across {len(data[0])} cells, sorted by |median r|")
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(outdir, f"3_day_corr_box_lead{lead:02d}.png"), dpi=110)
    plt.close(fig)


def summary_plot(med, feature_vars, leads, outdir):
    """median |r| vs lead, one line per variable. The plot that answers
    'which predictors survive to week 3+', which 43 separate boxes cannot."""
    fig, ax = plt.subplots(figsize=(11, 6))
    final = np.nanmean(med[-5:], axis=0)
    for k in np.argsort(-final):
        ax.plot(leads, med[:, k], lw=1.4, label=feature_vars[k])
    ax.set_xlabel("lead (days)")
    ax.set_ylabel("median |r| across cells")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7, ncol=2, loc="upper right")
    ax.set_title("Predictor-rain correlation decay by lead (anomaly space)")
    fig.tight_layout()
    fig.savefig(os.path.join(outdir, "3_day_summary_corr_vs_lead.png"), dpi=130)
    plt.close(fig)


def zone_heatmaps(zr, zones, feature_vars, leads, outdir):
    """zr: (n_lead, n_zone, n_feat). One heatmap per zone."""
    fig, axes = plt.subplots(1, len(zones), figsize=(5.5 * len(zones), 7), sharey=True)
    for zi, (ax, zname) in enumerate(zip(np.atleast_1d(axes), zones)):
        im = ax.imshow(zr[:, zi, :].T, aspect="auto", cmap="RdBu_r",
                       vmin=-0.8, vmax=0.8,
                       extent=[leads[0], leads[-1], len(feature_vars) - 0.5, -0.5])
        ax.set_yticks(range(len(feature_vars)))
        ax.set_yticklabels(feature_vars, fontsize=7)
        ax.set_xlabel("lead (days)")
        ax.set_title(zname)
    fig.colorbar(im, ax=axes, label="r (zone-mean anomaly vs zone-mean rain anomaly)")
    fig.savefig(os.path.join(outdir, "3_day_zone_corr_heatmap.png"), dpi=130,
                bbox_inches="tight")
    plt.close(fig)


def zone_map(cells, zlab, zones, outdir):
    """Sanity check: LOOK AT THIS. Box-based zones will have straight edges
    that cut through states. If that offends you, use the shapefile."""
    da = xr.DataArray(np.array([zones.index(z) for z in zlab], float),
                      coords={"cell": cells["cell"]}, dims="cell").unstack("cell")
    fig, ax = plt.subplots(figsize=(7, 7))
    im = ax.pcolormesh(da["lon"], da["lat"], da.values, cmap="tab10", vmin=0, vmax=9)
    cb = fig.colorbar(im, ax=ax, ticks=range(len(zones)))
    cb.ax.set_yticklabels(zones)
    ax.set_title("Zone assignment (APPROXIMATE -- not IMD's polygons)")
    ax.set_xlabel("lon"); ax.set_ylabel("lat")
    fig.tight_layout()
    fig.savefig(os.path.join(outdir, "3_day_zone_map.png"), dpi=130)
    plt.close(fig)


# ---------------- driver ----------------

if __name__ == "__main__":
    ecmwf_ds = ds_ecmv   # noqa: F821
    imd_ds = ds_imd      # noqa: F821

    os.makedirs(OUTDIR, exist_ok=True)
    t0 = time.perf_counter()

    with stage("Setup"):
        cells = build_cells(imd_ds)
        lat_pts = xr.DataArray(cells["lat"].values, dims="cell")
        lon_pts = xr.DataArray(cells["lon"].values, dims="cell")
        # zlab = assign_zones(cells)
        zones = [z for z, _ in ZONES_APPROX] if ZONE_SHAPEFILE is None \
            else sorted(set(zlab))
        for z in zones:
            print(f"    zone {z}: {(zlab == z).sum()} cells")

    with stage("Loading coarse ECMWF"):
        coarse = load_coarse(ecmwf_ds, imd_ds)
        feature_vars = list(coarse.data_vars)
        leads = (coarse["step"].values / np.timedelta64(1, "D")).astype(int)

    with stage("Loading IMD"):
        imd_cells = imd_ds[IMD_TARGET_VAR].stack(cell=("lat", "lon")).sel(
            cell=cells["cell"]).astype(DTYPE)
        with ProgressBar():
            imd_cells = imd_cells.compute()
        imd_cells = imd_cells.assign_coords(time=imd_cells["time"].dt.floor("D"))

    yrs = sorted(set(coarse["time"].dt.year.values.tolist()))
    tr_i = ~coarse["time"].dt.year.isin(yrs[-6:]).values   # climatology on train only

    n_z, n_f = len(zones), len(feature_vars)
    med = np.full((len(leads), n_f), np.nan)
    zr = np.full((len(leads), n_z, n_f), np.nan)
    zone_rain = np.full((len(leads), n_z, coarse.sizes["time"]), np.nan)

    with stage(f"Correlations: {len(leads)} leads x {n_f} vars"):
        for j, lead in enumerate(leads):
            s = coarse.isel(step=j)
            X = np.stack([s[v].interp(lat=lat_pts, lon=lon_pts).values
                          for v in feature_vars], axis=-1).astype(DTYPE)
            vt = s["valid_time"].dt.floor("D").values
            y = imd_cells.reindex(time=vt).values
            doy = xr.DataArray(vt, dims="t").dt.dayofyear.values

            if ANOMALY:
                # obs clim: DOY only. model clim: DOY x THIS lead (drift).
                co = _clim_grid(y[tr_i], doy[tr_i], CLIM_WINDOW_DAYS)
                y = y - co[doy - 1]
                for k in range(n_f):
                    cm = _clim_grid(X[tr_i, :, k], doy[tr_i], CLIM_WINDOW_DAYS)
                    X[:, :, k] -= cm[doy - 1]

            r = corr_cells(X, y)                      # (c, f)
            med[j] = np.nanmedian(np.abs(r), axis=0)
            boxplot_lead(r, feature_vars, int(lead), OUTDIR, ANOMALY)

            for zi, z in enumerate(zones):
                m = (zlab == z)
                yz = np.nanmean(y[:, m], axis=1)
                zone_rain[j, zi] = yz
                for k in range(n_f):
                    zr[j, zi, k] = corr_1d(np.nanmean(X[:, m, k], axis=1), yz)

            del X, y
            print(f"    lead {int(lead):>2} done", flush=True)

    with stage("Summary figures"):
        summary_plot(med, feature_vars, leads, OUTDIR)
        zone_heatmaps(zr, zones, feature_vars, leads, OUTDIR)
        zone_map(cells, zlab, zones, OUTDIR)

        # inter-zone rain correlation, per lead
        print(f"\n    Inter-zone rain-anomaly correlation (lead {leads[-1]}):")
        print("         " + "".join(f"{z:>7}" for z in zones))
        M = np.eye(n_z)
        for a in range(n_z):
            row = [corr_1d(zone_rain[-1, a], zone_rain[-1, b]) for b in range(n_z)]
            M[a] = row
            print(f"    {zones[a]:>5}" + "".join(f"{v:>7.2f}" for v in row))
        np.save(os.path.join(OUTDIR, "interzone_corr.npy"), M)

        xr.Dataset(
            {"median_abs_r": (("lead", "var"), med),
             "zone_r": (("lead", "zone", "var"), zr)},
            coords={"lead": leads, "var": feature_vars, "zone": zones},
        ).to_netcdf(os.path.join(OUTDIR, "correlations.nc"))

    print(f"\nTotal: {time.perf_counter() - t0:.1f}s")
    print(f"{len(leads)} box plots + summary + zone heatmap -> {OUTDIR}/")
    print("\nLook at summary_corr_vs_lead.png first -- 43 box plots are hard to")
    print("read as a set; the summary is what answers 'which predictors survive'.")

[ ] Setup ...
    4964 valid cells


NameError: name 'ZONE_SHAPEFILE' is not defined